In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, average_precision_score
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

In [2]:
# Read in cleaned dataset
X_with_urn = pd.read_csv('./model_data/final_features.csv') 
y = pd.read_csv('./model_data/labels.csv')  # 1 = target, 0 = non-target

entity_urns = X_with_urn['entity_urn'].copy()

X = X_with_urn.drop(columns=['entity_urn'])

In [3]:
# Separate targets and non-targets
targets = X[y.values == 1]   
non_targets = X[y.values == 0]  

In [4]:
# Split targets into Train/Val/Test manually
targets_train, targets_temp = train_test_split(
    targets, test_size=0.4, random_state=42
)
targets_val, targets_test = train_test_split(
    targets_temp, test_size=0.5, random_state=42
)

# Now split non-targets similarly
non_targets_train, non_targets_temp = train_test_split(
    non_targets, test_size=0.4, random_state=42
)
non_targets_val, non_targets_test = train_test_split(
    non_targets_temp, test_size=0.5, random_state=42
)

In [5]:
# Aggressively undersample non-targets (5x more non-targets than targets)
non_targets_train = resample(
    non_targets_train,
    replace=False,
    n_samples=len(targets_train) * 5,  # Adjust ratio here (5x, 10x etc.)
    random_state=42
)

In [6]:
# Combine targets and non-targets back
X_train = pd.concat([targets_train, non_targets_train])
y_train = pd.Series([1]*len(targets_train) + [0]*len(non_targets_train))

X_val = pd.concat([targets_val, non_targets_val])
y_val = pd.Series([1]*len(targets_val) + [0]*len(non_targets_val))

X_test = pd.concat([targets_test, non_targets_test])
y_test = pd.Series([1]*len(targets_test) + [0]*len(non_targets_test))

In [7]:
# # Shuffle each set (important so targets aren't clustered at top)
# For Train
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

train_data = X_train.copy()
train_data['label'] = y_train

train_data = train_data.sample(frac=1, random_state=42).reset_index(drop=True)

y_train = train_data['label']
X_train = train_data.drop(columns=['label'])

# # For Validation
X_val = X_val.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)

val_data = X_val.copy()
val_data['label'] = y_val
val_data = val_data.sample(frac=1, random_state=42).reset_index(drop=True)

y_val = val_data['label']
X_val = val_data.drop(columns=['label'])

# # For Test
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

test_data = X_test.copy()
test_data['label'] = y_test
test_data = test_data.sample(frac=1, random_state=42).reset_index(drop=True)

y_test = test_data['label']
X_test = test_data.drop(columns=['label'])

In [8]:
# Check
print("Train targets:", (y_train == 1).sum())
print("Validation targets:", (y_val == 1).sum())
print("Test targets:", (y_test == 1).sum())

print(f"Train size: {X_train.shape}, Validation size: {X_val.shape}, Test size: {X_test.shape}")

Train targets: 36
Validation targets: 12
Test targets: 13
Train size: (216, 30), Validation size: (2979, 30), Test size: (2980, 30)


In [9]:
from sklearn.model_selection import RandomizedSearchCV

# Define a parameter grid
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_features': ['sqrt', 'log2']
}

rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=20,
    scoring='average_precision',  # again, correct for your case
    n_jobs=-1,
    cv=3,
    verbose=2,
    random_state=42
)

random_search.fit(X_train, y_train)

# Best Model
print("Best Hyperparameters:", random_search.best_params_)
print(f"Best CV AP Score: {random_search.best_score_:.4f}")


Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   0.3s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=5, min_samples_split=10, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=5, min_samples_split=10, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=5, min_samples_split=10, n_estimators=100; total time=   0.2s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   0.3s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n

In [10]:
# model with best hyper params
best_rf = random_search.best_estimator_

In [11]:
y_val_pred_proba = best_rf.predict_proba(X_val)[:, 1]
ap_val = average_precision_score(y_val, y_val_pred_proba)

print(f"Validation AP Score (best model): {ap_val:.4f}")

Validation AP Score (best model): 0.9667


In [12]:
# Retrain on Full Training + Validation Set

# Combine training and validation sets
X_final_train = pd.concat([X_train, X_val])
y_final_train = pd.concat([y_train, y_val])

# Reinitialize the model with best hyperparameters
final_model = RandomForestClassifier(
    **random_search.best_params_,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Retrain
final_model.fit(X_final_train, y_final_train)

RandomForestClassifier(class_weight='balanced', max_depth=20,
                       min_samples_split=5, n_jobs=-1, random_state=42)

In [13]:
y_test_pred_proba = final_model.predict_proba(X_test)[:, 1]
ap_test = average_precision_score(y_test, y_test_pred_proba)

print(f"Test AP Score: {ap_test:.4f}")

Test AP Score: 1.0000


In [14]:
# Predict similarity scores
similarity_scores = final_model.predict_proba(X)[:, 1]

# Attach to company IDs
scored_companies = pd.DataFrame({
    'entity_urn': entity_urns,  # update with your real ID column
    'similarity_score': similarity_scores
})

scored_companies.sort_values(by='similarity_score', ascending=False, inplace=True)

# Save results
scored_companies.to_csv('company_similarity_scores.csv', index=False)


In [15]:
scored_companies

,entity_urn,similarity_score
19,urn:harmonic:company:9609784,1.000000
6,urn:harmonic:company:6585,1.000000
32,urn:harmonic:company:602158,1.000000
13,urn:harmonic:company:10356056,0.999899
10,urn:harmonic:company:2003862,0.990000
...,...,...
5309,urn:harmonic:company:2130939,0.000000
5310,urn:harmonic:company:2168330,0.000000
5311,urn:harmonic:company:2178656,0.000000
5312,urn:harmonic:company:2231339,0.000000


In [16]:
target_company_data = pd.read_parquet('./case_study_data/target_company_data.parquet')
target_company_data = target_company_data['entity_urn'].to_frame()

In [17]:
target_company_data

,entity_urn
0,urn:harmonic:company:1443095
1,urn:harmonic:company:1990249
2,urn:harmonic:company:3075425
3,urn:harmonic:company:10819970
4,urn:harmonic:company:3928206
...,...
56,urn:harmonic:company:1842794
57,urn:harmonic:company:48201423
58,urn:harmonic:company:773593
59,urn:harmonic:company:1869576


In [18]:
target_company_data.merge(scored_companies, how="left", on="entity_urn")

,entity_urn,similarity_score
0,urn:harmonic:company:1443095,0.949812
1,urn:harmonic:company:1990249,0.939899
2,urn:harmonic:company:3075425,0.970000
3,urn:harmonic:company:10819970,0.568691
4,urn:harmonic:company:3928206,0.929749
...,...,...
56,urn:harmonic:company:1842794,0.899003
57,urn:harmonic:company:48201423,0.674379
58,urn:harmonic:company:773593,0.829088
59,urn:harmonic:company:1869576,0.869547


In [19]:
print(np.min(similarity_scores), np.max(similarity_scores))

0.0 1.0
